In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DestinationLoss(nn.Module):
    def __init__(self, weight_main=0.5):
        super().__init__()
        self.weight_main = weight_main  # 위치 vs 운동 손실 간의 가중치 설정

    def forward(self, y_pred, y_true):
        # y_pred, y_true: [B, 1, 4] (lat, lon, SOG, COG)
        y_pred = y_pred.squeeze(1)
        y_true = y_true.squeeze(1)

        pred_lat, pred_lon = y_pred[:, 0], y_pred[:, 1]
        true_lat, true_lon = y_true[:, 0], y_true[:, 1]
        pred_sog, pred_cog = y_pred[:, 2], y_pred[:, 3]
        true_sog, true_cog = y_true[:, 2], y_true[:, 3]

        # ------------------------------------------------------------
        # 위치 예측 손실 (위도/경도)
        # → 기본적으로 MSE 사용
        # 참고 논문: 
        #   - "TrajectoryNet: An Embedded GPS Trajectory Representation for Point-based Deep Learning"
        #     (https://arxiv.org/abs/2108.04909)
        #     → GPS 기반 trajectory 예측 시 위치 오차는 일반적으로 MSE 사용
        # ------------------------------------------------------------
        loc_loss = F.mse_loss(pred_lat, true_lat) + F.mse_loss(pred_lon, true_lon)

        # ------------------------------------------------------------
        # SOG (속력)에 대해 Huber Loss (Smooth L1)
        # → MSE보다 이상치(outlier)에 덜 민감
        # 참고 논문:
        #   - "Fast and Accurate Deep Network Learning by Exponential Linear Units (ELUs)"
        #     (https://arxiv.org/abs/1511.07289)
        #     → Regression task에서 Huber loss는 안정적인 수렴에 효과적
        #   - 자율주행 관련:
        #     "MultiNet: Real-time Joint Semantic Reasoning for Autonomous Driving"
        #     (https://arxiv.org/abs/1612.07695)
        # ------------------------------------------------------------
        sog_loss = F.smooth_l1_loss(pred_sog, true_sog)

        # ------------------------------------------------------------
        # COG (방위각) 손실 - 각도 오차 처리
        # → 단순 MSE는 0° vs 360° 문제로 잘못된 손실 계산 발생
        # 해결책: Circular Loss 적용
        # 참고 논문:
        #   - "On Learning to Simulate Navigation Paths with Geodesic-Consistent Features"
        #     (https://arxiv.org/abs/2011.08258)
        #     → 방향/각도 오차 계산에 있어서 periodic/circular loss 활용
        #   - "Object Detection on Spherical Images using an Equirectangular Grid"
        #     (https://arxiv.org/abs/1805.08999)
        #     → 방위각 예측에서 각도 차이를 주기적으로 계산
        # ------------------------------------------------------------
        cog_diff = torch.remainder(pred_cog - true_cog + 180, 360) - 180
        cog_loss = torch.mean(cog_diff**2)

        motion_loss = sog_loss + cog_loss

        # ------------------------------------------------------------
        # 최종 손실 계산
        # → 위치와 운동 손실을 가중합
        # 참고 구조:
        #   - "Vessel Trajectory Prediction using Attention Mechanisms"
        #     (https://arxiv.org/abs/2106.02002)
        #     → 위치, 속도, 방향 등 여러 loss component를 가중합하는 방식 사용
        # ------------------------------------------------------------
        total_loss = self.weight_main * loc_loss + (1 - self.weight_main) * motion_loss

        return total_loss